# Phase 3 – Rhetorical Role Prediction (RR)
## BEST EXPERIMENT: LORMA Fine-Tuning on Architecture 4 (ALiBi + SwiGLU + RMSNorm)

**Technique**: **LORMA (Low-Rank Matrix Adaptation)** — Compulsory PEFT Technique  
**Base Model**: Phase-1 Architecture 4 – `ALiBiSwiGLULM` (Best Phase 1 Model, **Perplexity = 23.66**, ~105M params)  
**Decoder Checkpoint**: `/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt`  
**Custom Tokenizer**: `/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json`  
**Dataset**: Hugging Face `Exploration-Lab/CS-781-Capstone` (Phase_3_RR)  

---

### Why LORMA + Architecture 4 gives the Highest Score:
1. **Strongest Foundation**: Architecture 4 has the lowest perplexity (23.66) among all 4 Phase 1 models due to ALiBi relative attention bias and SwiGLU non-linearity.
2. **Compulsory LORMA**: Adapts both Attention projections (`q_proj`, `k_proj`, `v_proj`, `out_proj`) AND SwiGLU FFN (`gate_proj`, `up_proj`, `down_proj`).
3. **Rank-Stabilized Scaling**: Uses $\alpha / \sqrt{r}$ scaling to prevent gradient vanishing/explosion across deep layers.
4. **Per-Layer Learned Multipliers**: Dynamic adaptation weights per transformer block.
5. **Sentence-Chunking Engine**: Guarantees zero CUDA OOMs on long documents (up to 400+ sentences).
6. **Document-Level BiLSTM**: Captures document-wide discourse flow across sentence boundaries.

---
### Tuned-for-F1 version (model architecture untouched)
Only hyperparameters / training recipe were changed: higher LORMA ranks, separate LR for LORMA adapters vs. the new BiLSTM head, no weight decay on norm-like scalars, gradient accumulation over documents, fp16 GradScaler, longer schedule with early stopping, stronger class re-weighting, and dev-tuned per-class logit biases (decision thresholds) to maximise macro-F1.

## 1. Setup & Environment

In [ ]:
import os, gc, json, math, random, time, zipfile, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 2. Paths & Hugging Face Dataset Download

In [ ]:
from huggingface_hub import login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")

# User specified Phase 1 Model & Tokenizer paths
PHASE1_MODEL_PATH = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'
TOKENIZER_PATH    = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json'

# Phase 3 Dataset paths via Hugging Face download
REPO = "Exploration-Lab/CS-781-Capstone"
print("Downloading Phase 3 RR datasets from Hugging Face...")
TRAIN_CL = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/CL_train.json", repo_type="dataset", token=HF_TOKEN)
DEV_CL   = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/CL_dev.json", repo_type="dataset", token=HF_TOKEN)
TRAIN_IT = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/IT_train.json", repo_type="dataset", token=HF_TOKEN)
DEV_IT   = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/IT_dev.json", repo_type="dataset", token=HF_TOKEN)
LABEL_VOCAB_PATH = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/label_vocab.json", repo_type="dataset", token=HF_TOKEN)
TEST_CL  = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Test/CL_test.json", repo_type="dataset", token=HF_TOKEN)
TEST_IT  = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Test/IT_test.json", repo_type="dataset", token=HF_TOKEN)

OUTPUT_DIR = '/kaggle/working/phase3_arch4_lorma_best'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- LORMA Hyperparameters (tuned) ----------------
LORMA_RANK_ATTN = 32      # was 16 -> more adapter capacity
LORMA_RANK_FFN  = 16      # was 8
LORMA_ALPHA     = 32      # scaling = alpha / sqrt(rank)
LORMA_DROPOUT   = 0.10    # was 0.05 -> stronger regularisation for small RR dataset

# ---------------- Training Parameters (tuned) ----------------
MAX_SEQ_LEN     = 256
CONTEXT_WINDOW  = 3
CHUNK_SIZE      = 16      # Sub-batch size to guarantee no CUDA OOM
LR_LORMA        = 2e-4    # adapters on the pretrained backbone
LR_HEAD         = 1e-3    # freshly initialised BiLSTM + classifier learn faster
WEIGHT_DECAY    = 0.05
HEAD_DROPOUT    = 0.25    # dropout in BiLSTM head / classifier (was 0.1)
EPOCHS          = 20      # was 10; early stopping below prevents waste
PATIENCE        = 6       # stop if dev macro-F1 does not improve for 6 epochs
ACCUM_STEPS     = 4       # documents per optimizer step (effective batch = 4 docs)
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.05
CLASS_WEIGHT_POWER = 0.7  # was 0.5 -> boosts rare roles, which helps macro-F1
USE_BIAS_TUNING = True    # tune per-class logit offsets on dev for macro-F1

print("All paths & hyperparameters configured.")

## 3. Robust Checkpoint Loader (Auto-Fix Zip & `.format_version` Issues)

In [ ]:
def load_checkpoint_safe(model_path):
    """
    Safely loads PyTorch checkpoints even if:
    1. Path is a folder
    2. Archive has `.format_version` sitting at the root (inline_container.cc:180 fix)
    3. Archive has nested .pt files
    4. PyTorch 2.6+ weights_only default behavior
    """
    print(f"Inspecting checkpoint at: {model_path}")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Checkpoint not found at: {model_path}")

    # If directory, find the pt file inside
    if os.path.isdir(model_path):
        files = os.listdir(model_path)
        pt_files = [f for f in files if f.endswith('.pt') or f.endswith('.pth')]
        if pt_files:
            model_path = os.path.join(model_path, pt_files[0])
            print(f"Selected checkpoint file from folder: {model_path}")
        else:
            raise FileNotFoundError(f"No .pt file found in folder: {model_path}")

    # Try standard load first
    try:
        ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
        print("Loaded checkpoint directly.")
        return ckpt
    except Exception as e:
        err = str(e)
        print(f"Direct load encountered ({err[:90]}...). Attempting zip archive auto-fix...")

    # Handle zip archive issues (.format_version at root)
    extract_dir = "/kaggle/working/extracted_arch4_ckpt"
    fixed_pt    = "/kaggle/working/fixed_arch4_decoder_lm.pt"
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(model_path, 'r') as z:
        names = z.namelist()
        z.extractall(extract_dir)

    nested = [f for f in os.listdir(extract_dir) if f.endswith('.pt') and f != os.path.basename(model_path)]
    if nested:
        actual_path = os.path.join(extract_dir, nested[0])
        print(f"Found nested .pt: {actual_path}")
        return torch.load(actual_path, map_location='cpu', weights_only=False)

    # Re-pack into compliant PyTorch zip (everything under 'archive/')
    with zipfile.ZipFile(fixed_pt, 'w', compression=zipfile.ZIP_DEFLATED) as z_out:
        for root, _, files in os.walk(extract_dir):
            for file in files:
                if file in [".format_version", os.path.basename(fixed_pt)]:
                    continue
                full_path = os.path.join(root, file)
                rel_path  = os.path.relpath(full_path, extract_dir)
                arcname   = os.path.join("archive", rel_path) if not rel_path.startswith("archive") else rel_path
                z_out.write(full_path, arcname=arcname)

    print("Re-packed compliant archive successfully.")
    return torch.load(fixed_pt, map_location='cpu', weights_only=False)

checkpoint = load_checkpoint_safe(PHASE1_MODEL_PATH)
print("Checkpoint loaded successfully! Keys:", list(checkpoint.keys()) if isinstance(checkpoint, dict) else type(checkpoint))

## 4. Load Custom ByteBPETokenizer

In [ ]:
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []; self.merge_rank = {}; self.itos = []; self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        s = self._word_to_symbols(word)
        while len(s) > 1:
            p = [(s[i], s[i+1]) for i in range(len(s)-1)]
            c = [(self.merge_rank[x], i) for i, x in enumerate(p) if x in self.merge_rank]
            if not c: break
            _, i = min(c)
            s = s[:i] + [s[i]+s[i+1]] + s[i+2:]
        return s

    def encode(self, text, max_len=None):
        ids = [self.stoi.get(x, self.stoi['<unk>'])
               for w in text.split() for x in self._apply_merges_to_word(w)]
        return ids[:max_len] if max_len else ids

    def decode(self, ids):
        s = [self.itos[i] if i < len(self.itos) else '' for i in ids]
        words = [w for w in ''.join(s).split(END_OF_WORD) if w]
        return ' '.join(w.encode('latin-1').decode('utf-8', errors='replace') for w in words)

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']; t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}, PAD_ID: {PAD_ID}")

## 5. Robust Dataset Loader (Handles Train/Dev List & Test Dict Formats)

In [ ]:
def load_docs(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    elif isinstance(data, dict):
        if 'text' in data:
            return [data]
        docs = []
        for doc_id, body in data.items():
            if isinstance(body, list):
                docs.append({'id': doc_id, 'text': body})
            elif isinstance(body, dict):
                body['id'] = doc_id
                docs.append(body)
        return docs
    return []

with open(LABEL_VOCAB_PATH, 'r', encoding='utf-8') as f:
    lv = json.load(f)
LABELS = lv if isinstance(lv, list) else list(lv.keys())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

cl_train = load_docs(TRAIN_CL); cl_dev = load_docs(DEV_CL)
it_train = load_docs(TRAIN_IT); it_dev = load_docs(DEV_IT)
cl_test  = load_docs(TEST_CL);  it_test= load_docs(TEST_IT)

print(f"Official Rhetorical Roles ({NUM_LABELS}): {LABELS}")
print(f"Loaded documents -> CL Train: {len(cl_train)}, Dev: {len(cl_dev)}, Test: {len(cl_test)}")
print(f"Loaded documents -> IT Train: {len(it_train)}, Dev: {len(it_dev)}, Test: {len(it_test)}")

# Compute balanced class weights
all_train_lbls = [l for doc in cl_train + it_train for l in doc.get('labels', [])]
counts = Counter(all_train_lbls); total = sum(counts.values())
class_weights = torch.tensor(
    [(total / (NUM_LABELS * max(counts.get(l, 1), 1))) ** CLASS_WEIGHT_POWER for l in LABELS],
    dtype=torch.float32
).to(device)
class_weights = class_weights / class_weights.mean()

## 6. Architecture 4 Model Definition (ALiBi + SwiGLU + RMSNorm)

In [ ]:
@dataclass
class ALiBiConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2560
    dropout: float = 0.1

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads):
    def gsp2(n):
        s = 2 ** (-(2 ** -(math.log2(n) - 3)))
        return [s * (s**i) for i in range(n)]
    if math.log2(n_heads) == int(math.log2(n_heads)):
        return torch.tensor(gsp2(n_heads))
    cp2 = 2 ** math.floor(math.log2(n_heads))
    return torch.tensor(gsp2(cp2) + gsp2(2*cp2)[0::2][:n_heads-cp2])

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads; self.dh = d_model // n_heads; self.dropout = dropout
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.k_proj  = nn.Linear(d_model, d_model, bias=False)
        self.v_proj  = nn.Linear(d_model, d_model, bias=False)
        self.out_proj= nn.Linear(d_model, d_model, bias=False)
        self.q_norm  = RMSNorm(self.dh); self.k_norm = RMSNorm(self.dh)
        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer('slopes', slopes)

    def forward(self, x):
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        q = self.q_norm(q); k = self.k_norm(k)
        pq = torch.arange(S, device=x.device).unsqueeze(1)
        pk = torch.arange(S, device=x.device).unsqueeze(0)
        alibi = self.slopes * (pk - pq).unsqueeze(0)
        causal = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = alibi + causal.unsqueeze(0).unsqueeze(0)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1,2).contiguous().view(B, S, self.n_heads * self.dh))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class ALiBiBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1   = RMSNorm(cfg.d_model)
        self.attn = ALiBiCausalAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.n2   = RMSNorm(cfg.d_model)
        self.ffn  = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x

class ALiBiSwiGLULM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([ALiBiBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f  = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))
        for b in self.blocks: x = b(x)
        return self.norm_f(x)

## 7. LORMA Implementation (Attention + FFN Adaptation with Rank-Stabilization)

In [ ]:
class LORMALinear(nn.Module):
    """
    Low-Rank Matrix Adaptation (LORMA):
    - Targets Attention (q, k, v, out) and SwiGLU (gate, up, down)
    - Rank-stabilized denominator: alpha / sqrt(rank)
    - Per-layer learned dynamic scaling scalar (lorma_s)
    """
    def __init__(self, original: nn.Linear, rank: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        self.original = original
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / math.sqrt(rank)

        self.lorma_A = nn.Linear(original.in_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, original.out_features, bias=False)
        self.lorma_s = nn.Parameter(torch.ones(1))
        self.lorma_dropout = nn.Dropout(dropout)

        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)

        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

    def forward(self, x):
        base = self.original(x)
        delta = self.lorma_B(self.lorma_A(self.lorma_dropout(x))) * (self.lorma_s * self.scaling)
        return base + delta

def inject_lorma(model: nn.Module, attn_mods: List[str], ffn_mods: List[str],
                 r_attn: int, r_ffn: int, alpha: float, dropout: float):
    replaced = {'attn': 0, 'ffn': 0}
    for name, module in list(model.named_modules()):
        for target in attn_mods:
            if name.endswith(target) and isinstance(module, nn.Linear):
                p = model
                for part in name.split('.')[:-1]: p = getattr(p, part)
                setattr(p, name.split('.')[-1], LORMALinear(module, r_attn, alpha, dropout))
                replaced['attn'] += 1
                break
        for target in ffn_mods:
            if name.endswith(target) and isinstance(module, nn.Linear):
                p = model
                for part in name.split('.')[:-1]: p = getattr(p, part)
                setattr(p, name.split('.')[-1], LORMALinear(module, r_ffn, alpha, dropout))
                replaced['ffn'] += 1
                break
    print(f"Injected LORMA: {replaced['attn']} Attn layers (rank={r_attn}), {replaced['ffn']} FFN layers (rank={r_ffn})")

## 8. RR Classifier with OOM-Safe Sentence Chunking

In [ ]:
class RRClassifierArch4(nn.Module):
    def __init__(self, backbone: ALiBiSwiGLULM, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        self.num_labels = num_labels

        # Document-Level BiLSTM over the sentence sequence
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)

    def get_sent_embs(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        """
        Encodes sentences in small chunks (16 sentences at a time) to prevent CUDA OOM
        even on legal documents with 300+ sentences.
        """
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            c_ids  = input_ids[i : i + chunk_size]
            c_mask = attention_mask[i : i + chunk_size]
            h = self.backbone(c_ids) # [chunk, T, d_model]
            m = c_mask.unsqueeze(-1).float()
            emb = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        return torch.cat(sent_embs, dim=0) # [S, d_model]

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_size: int = 16):
        # 1. Chunked sentence encoding
        embs = self.get_sent_embs(input_ids, attention_mask, chunk_size=chunk_size) # [S, d_model]
        embs = self.dropout(embs)

        # 2. Document-level context
        x = embs.unsqueeze(0) # [1, S, d_model]
        lstm_out, _ = self.bilstm(x)
        lstm_out = self.dropout(lstm_out.squeeze(0)) # [S, d_model]

        # 3. Predict rhetorical role per sentence
        return self.classifier(lstm_out) # [S, num_labels]

## 9. Initialize Architecture 4 & Apply LORMA

In [ ]:
raw_cfg = checkpoint.get('config', {})
if hasattr(raw_cfg, '__dict__'): raw_cfg = raw_cfg.__dict__
d = raw_cfg if isinstance(raw_cfg, dict) else {}
cfg = ALiBiConfig(
    vocab_size=checkpoint.get('vocab_size', tokenizer.vocab_size),
    d_model=d.get('d_model', 768),
    n_layers=d.get('n_layers', 12),
    n_heads=d.get('n_heads', 12),
    d_ff=d.get('d_ff', 2560)
)

backbone = ALiBiSwiGLULM(cfg)
state_dict = checkpoint.get('model_state_dict', checkpoint)
backbone.load_state_dict(state_dict, strict=False)
print("Loaded pretrained Phase 1 weights into Architecture 4.")

# Freeze base backbone parameters
for p in backbone.parameters():
    p.requires_grad = False

# Apply LORMA to both Attention and SwiGLU FFN
ATTN_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'out_proj']
FFN_TARGETS  = ['gate_proj', 'up_proj', 'down_proj']
inject_lorma(backbone, ATTN_TARGETS, FFN_TARGETS, LORMA_RANK_ATTN, LORMA_RANK_FFN, LORMA_ALPHA, LORMA_DROPOUT)

model = RRClassifierArch4(backbone, NUM_LABELS, dropout=HEAD_DROPOUT).to(device)

tot = sum(p.numel() for p in model.parameters())
tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {tot:,} | Trainable Parameters: {tr:,} ({100*tr/tot:.2f}%)")

## 10. Document Dataset with ±3 Context Window

In [ ]:
class RRDocDataset(Dataset):
    def __init__(self, docs, tokenizer, label2id, max_seq_len, context_window=3, domain_prefix='', is_test=False):
        self.items = []
        pad_id = tokenizer.stoi.get('<pad>', 0)
        for doc in docs:
            sents = doc['text']
            labels = doc.get('labels', [None] * len(sents))
            doc_id = doc['id']
            ids_list, masks_list, lbls_list = [], [], []
            for i, (sent, lbl) in enumerate(zip(sents, labels)):
                s = max(0, i - context_window)
                e = min(len(sents), i + context_window + 1)
                ctx = f"{domain_prefix} " + " </s> ".join(sents[s:i])
                ctx += f" </s> {sent} </s> " + " </s> ".join(sents[i+1:e])
                ids = tokenizer.encode(ctx.strip(), max_len=max_seq_len)
                pad = max_seq_len - len(ids)
                ids_list.append(ids + [pad_id] * pad)
                masks_list.append([1] * len(ids) + [0] * pad)
                lbls_list.append(label2id.get(lbl, 0) if (not is_test and lbl) else -1)
            self.items.append({
                'doc_id': doc_id,
                'input_ids': torch.tensor(ids_list, dtype=torch.long),
                'attention_mask': torch.tensor(masks_list, dtype=torch.long),
                'labels': torch.tensor(lbls_list, dtype=torch.long),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

collate_fn = lambda b: b

cl_tr = RRDocDataset(cl_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]")
it_tr = RRDocDataset(it_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]")
cl_dv = RRDocDataset(cl_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]")
it_dv = RRDocDataset(it_dev,   tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]")
cl_te = RRDocDataset(cl_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]", is_test=True)
it_te = RRDocDataset(it_test,  tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]", is_test=True)

from torch.utils.data import ConcatDataset
train_ds = ConcatDataset([cl_tr, it_tr])
dev_ds   = ConcatDataset([cl_dv, it_dv])

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=1, shuffle=False, collate_fn=collate_fn)
print(f"Training Documents: {len(train_ds)} | Development Documents: {len(dev_ds)}")

## 11. Training with Mixed Precision & OOM Protection

In [ ]:
# ---------------- Optimizer: separate param groups ----------------
lorma_decay, lorma_nodecay, head_params = [], [], []
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    if 'lorma_s' in n:            lorma_nodecay.append(p)
    elif 'lorma_' in n:           lorma_decay.append(p)
    else:                         head_params.append(p)

optimizer = AdamW([
    {'params': lorma_decay,   'lr': LR_LORMA, 'weight_decay': WEIGHT_DECAY},
    {'params': lorma_nodecay, 'lr': LR_LORMA, 'weight_decay': 0.0},
    {'params': head_params,   'lr': LR_HEAD,  'weight_decay': WEIGHT_DECAY},
], betas=(0.9, 0.999))

n_batches = len(train_loader)
steps_per_epoch = math.ceil(n_batches / ACCUM_STEPS)
scheduler = OneCycleLR(
    optimizer,
    max_lr=[LR_LORMA, LR_LORMA, LR_HEAD],
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    pct_start=0.1,
    anneal_strategy='cos'
)

try:
    scaler = torch.amp.GradScaler('cuda')
except Exception:
    scaler = torch.cuda.amp.GradScaler()

trainable_params = [p for p in model.parameters() if p.requires_grad]

def evaluate(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            lbls = doc['labels'].to(device)
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, chunk_size=CHUNK_SIZE)
            all_p.extend(logits.argmax(-1).cpu().tolist())
            all_l.extend(lbls.cpu().tolist())
    vm = [i for i, l in enumerate(all_l) if l >= 0]
    ap = [all_p[i] for i in vm]; al = [all_l[i] for i in vm]
    return (f1_score(al, ap, average='macro', zero_division=0),
            precision_score(al, ap, average='macro', zero_division=0),
            recall_score(al, ap, average='macro', zero_division=0))

best_f1 = 0.0
bad_epochs = 0
best_ckpt = os.path.join(OUTPUT_DIR, 'best_arch4_lorma.pt')
print("Starting LORMA fine-tuning on Architecture 4...")
print(f"Docs/epoch: {n_batches} | optimizer steps/epoch: {steps_per_epoch} | effective batch: {ACCUM_STEPS} docs")

for epoch in range(EPOCHS):
    model.train()
    ep_loss, steps = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    has_grad = False
    for idx, batch in enumerate(train_loader):
        doc  = batch[0]
        ids  = doc['input_ids'].to(device)
        mask = doc['attention_mask'].to(device)
        lbls = doc['labels'].to(device)
        vm = lbls >= 0

        if vm.sum() > 0:
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, chunk_size=CHUNK_SIZE)
                loss = F.cross_entropy(logits[vm].float(), lbls[vm], weight=class_weights,
                                       label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss / ACCUM_STEPS).backward()
            has_grad = True
            ep_loss += loss.item()
            steps += 1

        if ((idx + 1) % ACCUM_STEPS == 0) or (idx + 1 == n_batches):
            if has_grad:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                has_grad = False
            optimizer.zero_grad(set_to_none=True)

    f1, p, r = evaluate(model, dev_loader)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {ep_loss/max(1,steps):.4f} | Dev Macro-F1: {f1:.4f} (P: {p:.4f}, R: {r:.4f})")
    if f1 > best_f1:
        best_f1 = f1; bad_epochs = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch+1, 'f1': f1}, best_ckpt)
        print(f"  -> Best model saved (Macro-F1: {best_f1:.4f})")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping: no improvement for {PATIENCE} epochs.")
            break

print(f"\nTraining complete! Peak Dev Macro-F1: {best_f1:.4f}")

## 12. Evaluation & Domain Breakdown

In [ ]:
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (dev macro-F1 {ckpt['f1']:.4f})")

def collect_logprobs(model, split_ds):
    """Returns (log-probs [N, C] as numpy, labels [N]) for all valid sentences."""
    loader = DataLoader(split_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    lps, ls = [], []
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, chunk_size=CHUNK_SIZE)
            lps.append(F.log_softmax(logits.float(), dim=-1).cpu().numpy())
            ls.append(doc['labels'].numpy())
    lps = np.concatenate(lps, 0); ls = np.concatenate(ls, 0)
    valid = ls >= 0
    return lps[valid], ls[valid]

def fast_macro_f1(y_true, y_pred, C):
    cm = np.bincount(y_true * C + y_pred, minlength=C * C).reshape(C, C)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    present = (cm.sum(1) > 0) | (cm.sum(0) > 0)   # matches sklearn (labels seen in y_true or y_pred)
    return f1[present].mean()

def tune_class_bias(lp, y, C, grid=np.linspace(-1.5, 1.5, 31), rounds=3):
    """Coordinate ascent on per-class additive log-prob offsets to maximise macro-F1."""
    bias = np.zeros(C)
    best = fast_macro_f1(y, (lp + bias).argmax(1), C)
    for _ in range(rounds):
        improved = False
        for c in range(C):
            cur = bias[c]; best_v = cur
            for v in grid:
                bias[c] = v
                s = fast_macro_f1(y, (lp + bias).argmax(1), C)
                if s > best + 1e-6:
                    best, best_v, improved = s, v, True
            bias[c] = best_v
        if not improved: break
    return bias, best

cl_lp, cl_y = collect_logprobs(model, cl_dv)
it_lp, it_y = collect_logprobs(model, it_dv)
all_lp = np.concatenate([cl_lp, it_lp], 0); all_y = np.concatenate([cl_y, it_y], 0)

base_f1 = fast_macro_f1(all_y, all_lp.argmax(1), NUM_LABELS)
BIAS_NP = np.zeros(NUM_LABELS)
if USE_BIAS_TUNING:
    BIAS_NP, tuned_f1 = tune_class_bias(all_lp, all_y, NUM_LABELS)
    print(f"Dev macro-F1 argmax: {base_f1:.4f} -> with tuned class biases: {tuned_f1:.4f}")
    # Shrink toward 0 by half-way if gain is tiny (guards against overfitting dev)
    if tuned_f1 - base_f1 < 0.002:
        BIAS_NP = np.zeros(NUM_LABELS)
        print("Gain too small -> biases disabled (kept plain argmax).")
    else:
        print("Biases:", {ID2LABEL[i]: round(float(b), 2) for i, b in enumerate(BIAS_NP)})
BIAS = torch.tensor(BIAS_NP, dtype=torch.float32, device=device)

cl_p = (cl_lp + BIAS_NP).argmax(1).tolist(); cl_l = cl_y.tolist()
it_p = (it_lp + BIAS_NP).argmax(1).tolist(); it_l = it_y.tolist()
cl_f1 = f1_score(cl_l, cl_p, average='macro', zero_division=0)
it_f1 = f1_score(it_l, it_p, average='macro', zero_division=0)
all_p = cl_p + it_p; all_l = cl_l + it_l

print("=== Full Classification Report (Combined Dev Set) ===")
print(classification_report(all_l, all_p, target_names=LABELS, zero_division=0))
print(f"Corporate Law (CL) Dev Macro-F1:       {cl_f1:.4f}")
print(f"Information Technology (IT) Dev Macro-F1: {it_f1:.4f}")
print(f"Overall Combined Dev Macro-F1:         {f1_score(all_l, all_p, average='macro'):.4f}")

## 13. Generate Test Predictions & Codabench Submission ZIP

In [ ]:
def gen_preds(model, test_ds, device, id2label):
    loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    preds = {}
    model.eval()
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids  = doc['input_ids'].to(device)
            mask = doc['attention_mask'].to(device)
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, chunk_size=CHUNK_SIZE)
            pred_ids = (F.log_softmax(logits.float(), dim=-1) + BIAS).argmax(-1).cpu().tolist()
            preds[doc['doc_id']] = [id2label[p] for p in pred_ids]
    return preds

cl_preds = gen_preds(model, cl_te, device, ID2LABEL)
it_preds = gen_preds(model, it_te, device, ID2LABEL)
all_preds = {**cl_preds, **it_preds}

# Verification: each document must have exactly len(text) predictions
errors = 0
for doc in cl_test + it_test:
    doc_id = doc['id']
    if len(all_preds.get(doc_id, [])) != len(doc['text']):
        errors += 1
print(f"Sanity check: {errors} sentence mismatch errors (Must be 0).")

# Save formatted predictions.json
pred_path = os.path.join(OUTPUT_DIR, 'predictions.json')
with open(pred_path, 'w', encoding='utf-8') as f:
    json.dump(all_preds, f, indent=2, ensure_ascii=False)
print(f"Predictions saved to: {pred_path}")

# Create submission.zip
zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname='predictions.json')
print(f"SUCCESS! CodaBench submission ready at: {zip_path}")